# Regression Models — Predicting Environmental Impact

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

In [ ]:
with open(os.path.join(DATA_DIR, 'product-category-benchmarks.json')) as f:
    benchmarks = json.load(f)

rows = []
for cat, data in benchmarks['categories'].items():
    rows.append({
        'category': cat,
        'co2e_median': data['co2eKg']['median'],
        'water_median': data['waterLiters']['median'],
        'energy_median': data['energyKwh']['median'],
        'price_median': data['typicalPrice']['median'],
        'lifespan_years': data['typicalLifespan']['years'],
        'weight_base': data['weightModel']['baseKg'],
        'n_materials': len(data.get('materialComposition', {})),
    })

df = pd.DataFrame(rows).set_index('category')
print(f'Dataset: {df.shape[0]} categories, {df.shape[1]} features')
df.head()

## Price -> CO2e Linear Regression

In [ ]:
X_price = df[['price_median']].values
y_co2e = df['co2e_median'].values

lr = LinearRegression()
lr.fit(X_price, y_co2e)
r2 = lr.score(X_price, y_co2e)

x_line = np.linspace(X_price.min(), X_price.max(), 100).reshape(-1, 1)
y_line = lr.predict(x_line)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['price_median'], df['co2e_median'], s=80, color='#2ecc71',
           edgecolors='white', zorder=3)

for i, cat in enumerate(df.index):
    ax.annotate(cat, (df['price_median'].iloc[i], df['co2e_median'].iloc[i]),
                fontsize=7, ha='left', va='bottom', alpha=0.7)

ax.plot(x_line, y_line, '--', color='#e74c3c', linewidth=2,
        label=f'Linear fit (R\u00b2={r2:.3f})')
ax.set_xlabel('Price (USD)')
ax.set_ylabel('CO\u2082e (kg)')
ax.set_title('Price vs. CO\u2082e — Linear Regression')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/price_co2e_regression.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Slope: {lr.coef_[0]:.4f} kgCO2e per USD')
print(f'Intercept: {lr.intercept_:.2f} kgCO2e')
print(f'R\u00b2: {r2:.4f}')

## Random Forest Model

In [ ]:
feature_cols = ['price_median', 'lifespan_years', 'weight_base', 'n_materials',
                'water_median', 'energy_median']
X_rf = df[feature_cols].values
y_rf = df['co2e_median'].values

rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=4)
rf.fit(X_rf, y_rf)

cv_folds = min(5, X_rf.shape[0])
cv_scores = cross_val_score(rf, X_rf, y_rf, cv=cv_folds, scoring='r2')
print(f'Cross-validation R\u00b2 scores: {cv_scores}')
print(f'Mean R\u00b2: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'Training R\u00b2: {rf.score(X_rf, y_rf):.3f}')

## Feature Importance

In [ ]:
perm_imp = permutation_importance(rf, X_rf, y_rf, n_repeats=30, random_state=42)

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std
}).sort_values('importance_mean', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp_df['feature'], imp_df['importance_mean'],
        xerr=imp_df['importance_std'], color='#3498db', edgecolor='white')
ax.set_xlabel('Permutation Importance (\u0394 R\u00b2)')
ax.set_title('Feature Importance — Random Forest CO\u2082e Prediction')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Model Comparison

| Model | Training R\u00b2 | Cross-Val R\u00b2 | Notes |
|-------|------------|--------------|-------|
| Linear (price only) | See above | — | Simple baseline |
| Random Forest | See above | See above | Captures nonlinear interactions |

The Random Forest model with multiple features generally outperforms the simple linear price-based
model, indicating that weight, lifespan, and material complexity contribute meaningful signal
beyond price alone.